## 4. Reproducible Processing Pipeline

### 4.1 Objective

The objective of this task is to create a reproducible processing pipeline connecting the UrbanSound8K dataset with the APAC audio compression codec developed in P2.

The pipeline uses a predefined audio recording and performs the following operations:

1. Locate the input recording.
2. Verify its integrity against the SHA-256 manifest from Task 3.
3. Load and normalize the audio using the corrected P2 WAV loader.
4. Compress the normalized audio using APAC.
5. Decompress the compressed representation.
6. Verify reconstruction correctness and calculate compression metrics.
7. Save the compressed file and processing results.

We use `quant_step = 1`, with optional DCT preprocessing disabled, to test lossless reconstruction of the normalized 16-bit PCM input.

The processing pipeline is designed to run through a single entry point and produce reproducible outputs.

The implementation reuses the codec from P2 rather than developing a separate compression method.

In [14]:

from pathlib import Path
import sys
import json
import hashlib
import inspect
import time
import os
import numpy as np

# Parent directory of P1, P2 and P3
PARENT_DIR = Path(
    r"C:\Users\HP\OneDrive\Dokumenter\Master\IDIG4120\P1"
)

# P3 notebook location (current working directory)
PROJECT_DIR = Path.cwd().resolve()

# UrbanSound8K dataset
DATASET_ROOT = PARENT_DIR / "archive"

# P2 repository
P2_DIR = PARENT_DIR / "idig4120_p2_codec"

# Manifest from Task 3
MANIFEST_PATH = PROJECT_DIR / "urbansound8k_manifest.json"

# Output directory
OUTPUT_DIR = PROJECT_DIR / "p3_output"

# Fixed test file from P2
TEST_RELATIVE_PATH = Path(
    "fold1/124489-9-0-12.wav"
)

def find_audio_root(dataset_root):
    candidates = [
        dataset_root,
        dataset_root / "audio",
        dataset_root / "UrbanSound8K" / "audio",
    ]

    for candidate in candidates:
        if (candidate / "fold1").is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not locate the UrbanSound8K fold directories"
    )

AUDIO_DIR = find_audio_root(DATASET_ROOT)

TEST_AUDIO = AUDIO_DIR / TEST_RELATIVE_PATH

# Check required inputs before processing
for label, path in [
    ("Dataset", DATASET_ROOT),
    ("P2 repository", P2_DIR),
    ("Manifest", MANIFEST_PATH),
    ("Test audio", TEST_AUDIO),
]:
    if not path.exists():
        raise FileNotFoundError(
            f"{label} not found: {path}"
        )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=== PROJECT SETUP ===")
print("P3 directory:", PROJECT_DIR)
print("Dataset root:", DATASET_ROOT)
print("P2 repository:", P2_DIR)
print("Manifest:", MANIFEST_PATH)
print("Test audio:", TEST_AUDIO)
print("Output directory:", OUTPUT_DIR)


=== PROJECT SETUP ===
P3 directory: C:\Users\HP\OneDrive\Dokumenter\Master\IDIG4120\P1\IDIG4120_DatasetProduct_P3
Dataset root: C:\Users\HP\OneDrive\Dokumenter\Master\IDIG4120\P1\archive
P2 repository: C:\Users\HP\OneDrive\Dokumenter\Master\IDIG4120\P1\idig4120_p2_codec
Manifest: C:\Users\HP\OneDrive\Dokumenter\Master\IDIG4120\P1\IDIG4120_DatasetProduct_P3\urbansound8k_manifest.json
Test audio: C:\Users\HP\OneDrive\Dokumenter\Master\IDIG4120\P1\archive\fold1\124489-9-0-12.wav
Output directory: C:\Users\HP\OneDrive\Dokumenter\Master\IDIG4120\P1\IDIG4120_DatasetProduct_P3\p3_output


In [15]:

# Make the P2 repository importable
sys.path.insert(0, str(P2_DIR.resolve()))

# Import the actual P2 functions
from codec import load_wav, compress, decompress

print("=== APAC CODEC IMPORTED ===")

print("load_wav:", inspect.signature(load_wav))
print("compress:", inspect.signature(compress))
print("decompress:", inspect.signature(decompress))


ModuleNotFoundError: No module named 'codec'

In [12]:

# Make the P2 repository importable
sys.path.insert(0, str(P2_DIR.resolve()))

# Import the actual P2 functions
from codec import load_wav, compress, decompress

print("=== APAC CODEC IMPORTED ===")

print("load_wav:", inspect.signature(load_wav))
print("compress:", inspect.signature(compress))
print("decompress:", inspect.signature(decompress))


ModuleNotFoundError: No module named 'codec'

In [16]:

def calculate_sha256(path):
    hasher = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            hasher.update(chunk)

    return hasher.hexdigest()


def verify_input(path, manifest_path, dataset_root):
    path = Path(path).resolve()
    dataset_root = Path(dataset_root).resolve()

    # Reject files outside the dataset
    relative_path = path.relative_to(
        dataset_root
    ).as_posix()

    with open(manifest_path, "r", encoding="utf-8") as f:
        manifest = json.load(f)

    entries = {
        entry["path"]: entry
        for entry in manifest["files"]
    }

    if relative_path not in entries:
        raise ValueError(
            f"File not found in manifest: {relative_path}"
        )

    if not path.is_file():
        raise FileNotFoundError(path)

    reference = entries[relative_path]

    actual_size = path.stat().st_size

    if actual_size != reference["size_bytes"]:
        raise ValueError(
            f"Size mismatch: {relative_path}"
        )

    actual_hash = calculate_sha256(path)

    if actual_hash != reference["sha256"]:
        raise ValueError(
            f"SHA-256 mismatch: {relative_path}"
        )

    return True


print("=== INPUT VERIFICATION ===")

verified = verify_input(
    TEST_AUDIO,
    MANIFEST_PATH,
    DATASET_ROOT
)

print("Input verification:", "PASS" if verified else "FAIL")


=== INPUT VERIFICATION ===
Input verification: PASS


In [17]:

# Lossless configuration
QUANT_STEP = 1


def load_audio(path):
    """Load audio using the P2 codec's corrected loader."""

    result = load_wav(path)

    if not isinstance(result, tuple) or len(result) != 2:
        raise TypeError(
            "load_wav must return (samples, sample_rate). "
            f"Actual return type: {type(result)}"
        )

    samples, sample_rate = result
    samples = np.asarray(samples)

    if samples.dtype != np.int16:
        raise ValueError(
            f"Expected int16, received {samples.dtype}"
        )

    if samples.ndim != 2:
        raise ValueError(
            f"Expected 2D samples, got shape {samples.shape}"
        )

    if samples.shape[1] < 1:
        raise ValueError("Audio has no channels")

    return samples, int(sample_rate)


def encode_audio(samples, sample_rate):
    """Call the P2 APAC encoder in lossless mode."""

    signature = inspect.signature(compress)
    parameters = signature.parameters

    kwargs = {}

    # Assign known parameter names
    if "quant_step" in parameters:
        kwargs["quant_step"] = QUANT_STEP
    elif "settings" in parameters:
        raise TypeError(
            "compress expects a settings object. "
            "The P2 settings format must be connected."
        )
    else:
        raise TypeError(
            "Unknown APAC quantization interface: "
            f"{signature}"
        )

    # Disable optional DCT preprocessing where exposed
    if "dct_mode" in parameters:
        kwargs["dct_mode"] = 0

    if "dct_param" in parameters:
        kwargs["dct_param"] = 0

    compressed = compress(
        samples,
        sample_rate,
        **kwargs
    )

    if not isinstance(compressed, (bytes, bytearray)):
        raise TypeError(
            "Expected APAC bytes, got "
            f"{type(compressed)}"
        )

    return bytes(compressed)


def decode_audio(compressed_bytes):
    """Decode the APAC container using the P2 decoder."""

    result = decompress(compressed_bytes)

    # Some codecs return (samples, sample_rate)
    if isinstance(result, tuple):
        if len(result) != 2:
            raise TypeError(
                f"Unexpected decoder output: {len(result)} items"
            )

        samples, decoded_sample_rate = result
    else:
        samples = result

    return np.asarray(samples)


In [18]:

def run_pipeline():
    print("=== APAC REPRODUCIBLE PIPELINE ===")

    # 1. Verify input
    print("\n1. Verifying input...")

    verify_input(
        TEST_AUDIO,
        MANIFEST_PATH,
        DATASET_ROOT
    )

    print("Input integrity: PASS")

    # 2. Load audio
    print("\n2. Loading audio...")

    samples, sample_rate = load_audio(TEST_AUDIO)

    print("Shape:", samples.shape)
    print("Data type:", samples.dtype)
    print("Sample rate:", sample_rate)

    # 3. Compress
    print("\n3. Compressing with APAC...")

    start = time.perf_counter()

    compressed = encode_audio(
        samples,
        sample_rate
    )

    encode_time = time.perf_counter() - start

    # 4. Decompress
    print("\n4. Decompressing APAC...")

    start = time.perf_counter()

    reconstructed = decode_audio(compressed)

    decode_time = time.perf_counter() - start

    # 5. Evaluate reconstruction
    print("\n5. Evaluating reconstruction...")

    shape_match = reconstructed.shape == samples.shape
    dtype_match = reconstructed.dtype == samples.dtype

    bit_exact = (
        shape_match
        and dtype_match
        and np.array_equal(samples, reconstructed)
    )

    if not shape_match or not dtype_match:
        raise AssertionError(
            "Reconstructed audio shape/dtype mismatch"
        )

    difference = (
        samples.astype(np.float64)
        - reconstructed.astype(np.float64)
    )

    rmse = float(
        np.sqrt(np.mean(difference ** 2))
    )

    if not bit_exact:
        raise AssertionError(
            "Lossless APAC round trip was not bit-exact"
        )

    # 6. Calculate compression statistics
    pcm_bytes = int(samples.nbytes)
    apac_bytes = len(compressed)

    compression_ratio = pcm_bytes / apac_bytes

    size_reduction_percent = (
        1 - apac_bytes / pcm_bytes
    ) * 100

    results = {
        "input_file": TEST_RELATIVE_PATH.as_posix(),
        "sample_rate": sample_rate,
        "shape": list(samples.shape),
        "dtype": str(samples.dtype),
        "quant_step": QUANT_STEP,
        "dct_enabled": False,
        "pcm_bytes": pcm_bytes,
        "apac_bytes": apac_bytes,
        "compression_ratio": compression_ratio,
        "size_reduction_percent": size_reduction_percent,
        "bit_exact": bool(bit_exact),
        "rmse": rmse,
        "encode_time_seconds": encode_time,
        "decode_time_seconds": decode_time,
    }

    # 7. Save outputs
    print("\n6. Saving outputs...")

    compressed_path = OUTPUT_DIR / "compressed.apac"
    results_path = OUTPUT_DIR / "results.json"

    compressed_path.write_bytes(compressed)

    with open(
        results_path,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(results, f, indent=2)

    print("\n=== PIPELINE RESULTS ===")

    for key, value in results.items():
        print(f"{key}: {value}")

    print("\nPipeline completed successfully.")

    return results


In [19]:

results = run_pipeline()


=== APAC REPRODUCIBLE PIPELINE ===

1. Verifying input...
Input integrity: PASS

2. Loading audio...


NameError: name 'load_wav' is not defined